In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data")

transactions = pd.read_parquet(
    DATA_DIR / "online_retail_II_cleaned.parquet"
)

print(transactions.shape)
display(transactions.head())
display(transactions.dtypes)

(779425, 11)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancelled_invoice,line_value
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010,False,30.0


Invoice                 string[python]
StockCode               string[python]
Description             string[python]
Quantity                         int64
InvoiceDate             datetime64[ns]
Price                          float64
Customer ID             string[python]
Country                 string[python]
source_sheet            string[python]
is_cancelled_invoice              bool
line_value                     float64
dtype: object

In [2]:
assert transactions["Customer ID"].notna().all()
assert transactions["Quantity"].gt(0).all()
assert transactions["Price"].gt(0).all()
assert not transactions["is_cancelled_invoice"].any()
assert not transactions.duplicated().any()

In [3]:
non_merchandise_codes = {
    "M",             # Manual entry
    "ADJUST",        # Accounting adjustment
    "ADJUST2",       # Accounting adjustment
    "POST",          # Postage
    "DOT",           # Dotcom postage
    "BANK CHARGES",  # Bank charges
    "AMAZONFEE",     # Amazon fees
    "B",             # Bad-debt adjustment
    "C2",            # Carriage
    "CRUK",          # Commission
    "D",             # Discount
    "PADS",          # Administrative/sample entry
    "S",             # Sample
    "TEST001",       # Test transaction
    "TEST002",       # Test transaction
}

transactions["normalised_stock_code"] = (
    transactions["StockCode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

transactions["is_merchandise"] = (
    ~transactions["normalised_stock_code"]
    .isin(non_merchandise_codes)
)

In [4]:
non_merchandise_summary = (
    transactions.loc[~transactions["is_merchandise"]]
    .groupby(
        ["normalised_stock_code", "Description"],
        dropna=False
    )
    .agg(
        rows=("Invoice", "size"),
        invoices=("Invoice", "nunique"),
        customers=("Customer ID", "nunique"),
        total_value=("line_value", "sum"),
        maximum_price=("Price", "max"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(non_merchandise_summary)

,normalised_stock_code,Description,rows,invoices,customers,total_value,maximum_price
9,POST,POSTAGE,1803,1803,405,124648.040,8142.750
7,M,Manual,681,620,438,151777.670,10953.500
4,C2,CARRIAGE,248,248,45,12546.000,150.000
3,BANK CHARGES,Bank Charges,31,30,19,450.001,15.000
0,ADJUST,Adjustment by john on 26/01/2010 16,18,18,15,2107.410,342.800
8,PADS,PADS TO MATCH ALL CUSHIONS,17,17,15,0.017,0.001
6,DOT,DOTCOM POSTAGE,16,16,1,11906.360,1599.260
1,ADJUST,Adjustment by john on 26/01/2010 17,14,14,11,1431.110,387.540
10,TEST001,This is a test product.,9,9,2,225.000,4.500
5,D,Discount,5,5,5,397.890,101.990


In [5]:
print(
    "Non-merchandise lines:",
    (~transactions["is_merchandise"]).sum()
)

print(
    "Percentage of transaction lines:",
    (~transactions["is_merchandise"]).mean()
)

Non-merchandise lines: 2846
Percentage of transaction lines: 0.0036514096930429485


In [6]:
merchandise = (
    transactions.loc[transactions["is_merchandise"]]
    .copy()
)

print("All cleaned transaction lines:", len(transactions))
print("Merchandise transaction lines:", len(merchandise))

All cleaned transaction lines: 779425
Merchandise transaction lines: 776579


In [7]:
merchandise["order_id"] = (
    merchandise["Invoice"]
    .astype("string")
    .str.strip()
)

In [8]:
order_integrity = (
    merchandise
    .groupby("order_id")
    .agg(
        customers=("Customer ID", "nunique"),
        timestamps=("InvoiceDate", "nunique"),
        source_sheets=("source_sheet", "nunique"),
    )
)

display(order_integrity["customers"].value_counts())
display(order_integrity["timestamps"].value_counts())
display(order_integrity["source_sheets"].value_counts())

assert order_integrity["customers"].max() == 1

multi_timestamp_invoices = (
    order_integrity["timestamps"] > 1
).sum()

print(
    "Invoices containing multiple timestamps:",
    multi_timestamp_invoices
)

customers
1    36594
Name: count, dtype: int64

timestamps
1    36530
2       64
Name: count, dtype: int64

source_sheets
1    36594
Name: count, dtype: int64

Invoices containing multiple timestamps: 64


In [9]:
order_timestamp_span = (
    merchandise
    .groupby("order_id")["InvoiceDate"]
    .agg(["min", "max"])
)

order_timestamp_span["span_minutes"] = (
    order_timestamp_span["max"]
    - order_timestamp_span["min"]
).dt.total_seconds() / 60

print(
    "Maximum within-invoice timestamp span:",
    order_timestamp_span["span_minutes"].max(),
    "minutes"
)

assert order_timestamp_span["span_minutes"].max() <= 2

Maximum within-invoice timestamp span: 2.0 minutes


### Invoices with multiple timestamps

Sixty-four invoice numbers contained product lines recorded across two
timestamps. All belonged to a single customer, and the maximum difference
within an invoice was two minutes. These records were therefore treated as
single orders, using the earliest timestamp as the order time and aggregating
all product lines under the invoice number.

In [10]:
customers_per_order = (
    merchandise.groupby("order_id")["Customer ID"].nunique()
)

customers_per_order.value_counts()

Customer ID
1    36594
Name: count, dtype: int64

In [11]:
assert customers_per_order.max() == 1

In [12]:
dates_per_order = (
    merchandise.groupby("order_id")["InvoiceDate"].nunique()
)

dates_per_order.value_counts()

InvoiceDate
1    36530
2       64
Name: count, dtype: int64

In [13]:
orders = (
    merchandise
    .groupby("order_id", as_index=False)
    .agg(
        customer_id=("Customer ID", "first"),
        order_date=("InvoiceDate", "min"),
        order_value=("line_value", "sum"),
        total_quantity=("Quantity", "sum"),
        unique_products=("StockCode", "nunique"),
        product_lines=("StockCode", "size"),
        country=("Country", "first"),
    )
)

In [14]:
assert len(orders) == len(order_integrity)

print(
    "Order count agrees with integrity table:",
    len(orders)
)

Order count agrees with integrity table: 36594


In [15]:
orders["average_item_price"] = (
    orders["order_value"] / orders["total_quantity"]
)

In [16]:
print(f"All cleaned transaction lines: {len(transactions):,}")
print(f"Merchandise transaction lines: {len(merchandise):,}")
print(f"Valid merchandise orders: {len(orders):,}")
print(f"Customers: {orders['customer_id'].nunique():,}")

display(orders.head())
display(orders.describe())

All cleaned transaction lines: 779,425
Merchandise transaction lines: 776,579
Valid merchandise orders: 36,594
Customers: 5,852


,order_id,customer_id,order_date,order_value,total_quantity,unique_products,product_lines,country,average_item_price
0,489434,13085,2009-12-01 07:45:00,505.30,166,8,8,United Kingdom,3.043976
1,489435,13085,2009-12-01 07:46:00,145.80,60,4,4,United Kingdom,2.430000
2,489436,13078,2009-12-01 09:06:00,630.33,193,19,19,United Kingdom,3.265959
3,489437,15362,2009-12-01 09:08:00,310.75,145,23,23,United Kingdom,2.143103
4,489438,18102,2009-12-01 09:24:00,2286.24,826,17,17,United Kingdom,2.767845


,order_date,order_value,total_quantity,unique_products,product_lines,average_item_price
count,36594,36594.000000,36594.000000,36594.000000,36594.000000,36594.000000
mean,2010-12-27 13:37:21.231349248,466.431183,286.897797,20.934607,21.221484,2.569704
min,2009-12-01 07:45:00,0.380000,1.000000,1.000000,1.000000,0.040000
25%,2010-06-30 08:43:15,158.630000,74.000000,7.000000,7.000000,1.412448
50%,2010-12-01 16:15:30,302.550000,153.000000,15.000000,15.000000,1.902500
75%,2011-07-13 12:22:30,473.880000,288.000000,27.000000,28.000000,2.633149
max,2011-12-09 12:50:00,168469.600000,87167.000000,540.000000,541.000000,649.500000
std,NaN,1358.869317,1238.200490,22.403772,22.969693,7.974263


In [17]:
assert orders["order_id"].is_unique
assert orders["customer_id"].notna().all()
assert orders["order_value"].gt(0).all()
assert orders["total_quantity"].gt(0).all()
assert orders["unique_products"].ge(1).all()

In [18]:
orders.nlargest(
    20, "order_value"
)[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_value",
        "total_quantity",
        "unique_products",
        "country",
    ]
]

,order_id,customer_id,order_date,order_value,total_quantity,unique_products,country
36561,581483,16446,2011-12-09 09:15:00,168469.60,80995,1,United Kingdom
20095,541431,12346,2011-01-18 10:01:00,77183.60,74215,1,United Kingdom
1587,493819,14156,2010-01-07 12:34:00,44051.60,25018,94,EIRE
26071,556444,15098,2011-06-10 15:28:00,38970.00,60,1,United Kingdom
13228,524181,17450,2010-09-27 16:59:00,33167.80,8172,13,United Kingdom
30526,567423,17450,2011-09-20 11:05:00,31698.16,12572,12,United Kingdom
14411,526934,18102,2010-10-14 09:46:00,26007.08,5079,15,United Kingdom
9877,515944,18102,2010-07-15 15:29:00,22863.36,4992,17,United Kingdom
26257,556917,12415,2011-06-15 13:37:00,22775.93,15049,138,Australia
32545,572209,18102,2011-10-21 12:08:00,22206.00,1920,7,United Kingdom


In [19]:
orders = orders.sort_values(
    ["customer_id", "order_date", "order_id"]
).reset_index(drop=True)

orders["order_number"] = (
    orders.groupby("customer_id").cumcount() + 1
)

In [20]:
repeat_customer_ids = (
    orders.groupby("customer_id")
          .size()
          .loc[lambda x: x >= 2]
          .index[:5]
)

orders[
    orders["customer_id"].isin(repeat_customer_ids)
].head(20)

,order_id,customer_id,order_date,order_value,total_quantity,unique_products,product_lines,country,average_item_price,order_number
0,499763,12346,2010-03-02 13:08:00,27.05,5,5,5,United Kingdom,5.410000,1
1,513774,12346,2010-06-28 13:53:00,142.31,19,19,19,United Kingdom,7.490000,2
2,541431,12346,2011-01-18 10:01:00,77183.60,74215,1,1,United Kingdom,1.040000,3
3,529924,12347,2010-10-31 14:20:00,611.53,509,40,40,Iceland,1.201434,1
4,537626,12347,2010-12-07 14:57:00,711.79,319,31,31,Iceland,2.231317,2
5,542237,12347,2011-01-26 14:30:00,475.39,315,29,29,Iceland,1.509175,3
6,549222,12347,2011-04-07 10:43:00,636.25,483,24,24,Iceland,1.317288,4
7,556201,12347,2011-06-09 13:01:00,382.52,196,18,18,Iceland,1.951633,5
8,562032,12347,2011-08-02 08:48:00,584.91,277,22,22,Iceland,2.111588,6
9,573511,12347,2011-10-31 12:25:00,1294.32,676,47,47,Iceland,1.914675,7


In [21]:
first_orders = (
    orders.loc[orders["order_number"] == 1]
    .copy()
    .rename(columns={
        "order_id": "first_order_id",
        "order_date": "first_order_date",
        "order_value": "first_order_value",
        "total_quantity": "first_order_quantity",
        "unique_products": "first_order_unique_products",
        "product_lines": "first_order_product_lines",
        "average_item_price": "first_order_average_item_price",
        "country": "first_order_country",
    })
)

In [22]:
first_orders = first_orders[
    [
        "customer_id",
        "first_order_id",
        "first_order_date",
        "first_order_value",
        "first_order_quantity",
        "first_order_unique_products",
        "first_order_product_lines",
        "first_order_average_item_price",
        "first_order_country",
    ]
]

In [23]:
first_orders["first_order_weekday"] = (
    first_orders["first_order_date"].dt.day_name()
)

first_orders["first_order_month"] = (
    first_orders["first_order_date"].dt.month
)

first_orders["first_order_hour"] = (
    first_orders["first_order_date"].dt.hour
)

first_orders["first_order_is_weekend"] = (
    first_orders["first_order_date"].dt.dayofweek >= 5
)

In [24]:
later_order_candidates = orders.merge(
    first_orders[
        ["customer_id", "first_order_date"]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

later_order_candidates = later_order_candidates[
    later_order_candidates["order_date"]
    > later_order_candidates["first_order_date"]
]

second_orders = (
    later_order_candidates
    .groupby("customer_id", as_index=False)
    .agg(
        second_order_date=("order_date", "min")
    )
)

In [25]:
customers = first_orders.merge(
    second_orders,
    on="customer_id",
    how="left",
    validate="one_to_one",
)

In [26]:
customers["days_to_second_order"] = (
    customers["second_order_date"]
    - customers["first_order_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [27]:
assert customers[
    "days_to_second_order"
].dropna().gt(0).all()

In [28]:
customers[
    [
        "customer_id",
        "first_order_date",
        "second_order_date",
        "days_to_second_order",
    ]
].head(20)

,customer_id,first_order_date,second_order_date,days_to_second_order
0,12346,2010-03-02 13:08:00,2010-06-28 13:53:00,118.031250
1,12347,2010-10-31 14:20:00,2010-12-07 14:57:00,37.025694
2,12348,2010-09-27 14:59:00,2010-12-16 19:09:00,80.173611
3,12349,2010-04-29 13:20:00,2010-10-28 08:23:00,181.793750
4,12350,2011-02-02 16:01:00,NaT,NaN
5,12351,2010-11-29 15:23:00,NaT,NaN
6,12352,2010-11-12 10:20:00,2010-11-29 10:07:00,16.990972
7,12353,2010-10-27 12:44:00,2011-05-19 17:47:00,204.210417
8,12354,2011-04-21 13:11:00,NaT,NaN
9,12355,2010-05-21 11:59:00,2011-05-09 13:49:00,353.076389


In [29]:
data_end_date = transactions["InvoiceDate"].max()

data_end_date

Timestamp('2011-12-09 12:50:00')

In [30]:
customers["available_followup_days"] = (
    data_end_date - customers["first_order_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [31]:
customers["has_full_90d_followup"] = (
    customers["available_followup_days"] >= 90
)

In [32]:
customers["has_full_90d_followup"].value_counts()

has_full_90d_followup
True     5256
False     596
Name: count, dtype: int64

In [33]:
eligible_customers = (
    customers.loc[customers["has_full_90d_followup"]]
    .copy()
)

In [34]:
eligible_customers["repeat_within_90d"] = (
    eligible_customers["days_to_second_order"]
    .between(0, 90, inclusive="right")
    .astype(int)
)

1 = second valid order occurred within 90 days
0 = no second valid order occurred within 90 days

In [35]:
eligible_customers["repeat_within_90d"].value_counts()
eligible_customers["repeat_within_90d"].value_counts(normalize=True)

repeat_within_90d
0    0.534817
1    0.465183
Name: proportion, dtype: float64

In [36]:
eligible_customers.loc[
    eligible_customers["repeat_within_90d"] == 1,
    [
        "customer_id",
        "first_order_date",
        "second_order_date",
        "days_to_second_order",
    ]
].head()

,customer_id,first_order_date,second_order_date,days_to_second_order
1,12347,2010-10-31 14:20:00,2010-12-07 14:57:00,37.025694
2,12348,2010-09-27 14:59:00,2010-12-16 19:09:00,80.173611
6,12352,2010-11-12 10:20:00,2010-11-29 10:07:00,16.990972
10,12356,2010-10-11 09:42:00,2010-11-11 14:23:00,31.195139
13,12359,2009-12-05 13:32:00,2009-12-16 15:24:00,11.077778


In [37]:
eligible_customers.loc[
    eligible_customers["repeat_within_90d"] == 0,
    [
        "customer_id",
        "first_order_date",
        "second_order_date",
        "days_to_second_order",
    ]
].head()

,customer_id,first_order_date,second_order_date,days_to_second_order
0,12346,2010-03-02 13:08:00,2010-06-28 13:53:00,118.031250
3,12349,2010-04-29 13:20:00,2010-10-28 08:23:00,181.793750
4,12350,2011-02-02 16:01:00,NaT,NaN
5,12351,2010-11-29 15:23:00,NaT,NaN
7,12353,2010-10-27 12:44:00,2011-05-19 17:47:00,204.210417


In [38]:
assert eligible_customers["customer_id"].is_unique
assert eligible_customers["available_followup_days"].ge(90).all()
assert eligible_customers["repeat_within_90d"].isin([0, 1]).all()

assert (
    eligible_customers.loc[
        eligible_customers["repeat_within_90d"] == 1,
        "days_to_second_order"
    ].le(90).all()
)

In [39]:
feature_columns = [
    "first_order_value",
    "first_order_quantity",
    "first_order_unique_products",
    "first_order_product_lines",
    "first_order_average_item_price",
    "first_order_country",
    "first_order_weekday",
    "first_order_month",
    "first_order_hour",
    "first_order_is_weekend",
]

eligible_customers[feature_columns].head()

,first_order_value,first_order_quantity,first_order_unique_products,first_order_product_lines,first_order_average_item_price,first_order_country,first_order_weekday,first_order_month,first_order_hour,first_order_is_weekend
0,27.05,5,5,5,5.410000,United Kingdom,Tuesday,3,13,False
1,611.53,509,40,40,1.201434,Iceland,Sunday,10,14,True
2,221.16,372,19,19,0.594516,Finland,Monday,9,14,False
3,1068.52,473,46,46,2.259027,Italy,Thursday,4,13,False
4,294.40,196,16,16,1.502041,Norway,Wednesday,2,16,False


In [40]:
output_path = DATA_DIR / "customer_retention_dataset.parquet"

eligible_customers.to_parquet(
    output_path,
    index=False,
)

check = pd.read_parquet(output_path)

assert check.shape == eligible_customers.shape

print(f"Saved {len(check):,} eligible customers")

Saved 5,256 eligible customers


In [41]:
print("Orders:", len(orders))
print("Customers before censoring:", len(customers))
print("Eligible customers:", len(eligible_customers))
print(
    "90-day repeat rate:",
    eligible_customers["repeat_within_90d"].mean()
)
print(
    "Exact zero-day second orders:",
    customers["days_to_second_order"].eq(0).sum()
)

print(
    "Second orders within 24 hours:",
    customers["days_to_second_order"]
    .between(0, 1, inclusive="both")
    .sum()
)

Orders: 36594
Customers before censoring: 5852
Eligible customers: 5256
90-day repeat rate: 0.4651826484018265
Exact zero-day second orders: 0
Second orders within 24 hours: 304


In [42]:
print("Merchandise orders:", len(orders))
print(
    "Maximum average item price:",
    orders["average_item_price"].max()
)
print(
    "Maximum first-order average item price:",
    first_orders["first_order_average_item_price"].max()
)

Merchandise orders: 36594
Maximum average item price: 649.5
Maximum first-order average item price: 295.0


## Customer-dataset summary

- **Valid merchandise orders:** 36,594 distinct non-cancelled invoices
  containing at least one merchandise line.
- **Customers before censoring:** 5,852 identifiable customers with at
  least one valid merchandise order.
- **Eligible customers:** 5,256 customers whose first merchandise order
  had at least 90 days of follow-up.
- **Excluded by right censoring:** 596 customers.
- **90-day repeat rate:** 46.52%.
- **Exact simultaneous second orders:** 0 after defining the second order
  as the earliest order strictly after the first-order timestamp.

### Non-merchandise transactions

Administrative transaction codes, including manual adjustments, postage,
bank charges, discounts, commissions, carriage charges, samples, and test
records, were excluded from purchase-order construction.

The original records remain in the cleaned transaction dataset for
auditability. Invoices containing both merchandise and administrative
lines were retained, but basket features were calculated from merchandise
lines only. Invoices containing no merchandise were not treated as
purchases.